In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp05/ex_7.py ──
"""
Shared infrastructure for Exercise 7 — Transfer Learning.

Contains: CIFAR-10 data loading, feature visualisation helpers,
ExperimentTracker/ModelRegistry setup, training harness, and the serving
helpers (flat-row ONNX adapter, request records, ONNX artifact attach).
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as T

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.engines.model_registry import LocalFileArtifactStore
from kailash_ml.types import MetricSpec


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

OUTPUT_DIR = Path("outputs") / "ex7_transfer_learning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The ModelRegistry stores artifact files (model.pkl, model.onnx) in an
# ArtifactStore. We construct the store explicitly so Part 5 can attach an
# ONNX serving artifact to a registered version (see attach_onnx_artifact).
ARTIFACT_STORE = LocalFileArtifactStore(".kailash_ml/artifacts")

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — CIFAR-10 (full 50K, resized for ResNet-18)
# ════════════════════════════════════════════════════════════════════════
# ResNet-18 was designed for 224x224 ImageNet images. CIFAR-10 is 32x32.
# We resize to 96x96: ResNet's strided convolutions shrink spatial dims
# by 32x, so 32x32 would collapse to 1x1 before final pooling. 96x96
# gives a 3x3 final feature map — enough spatial information to learn.

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "cifar10"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_SIZE = 96
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
N_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 8

CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

# Standard transforms: ImageNet normalisation + resize for ResNet
train_transform = T.Compose(
    [
        T.Resize((INPUT_SIZE, INPUT_SIZE)),
        T.RandomHorizontalFlip(),
        T.RandomCrop(INPUT_SIZE, padding=8),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)
val_transform = T.Compose(
    [
        T.Resize((INPUT_SIZE, INPUT_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


def load_cifar10() -> tuple[
    torchvision.datasets.CIFAR10,
    torchvision.datasets.CIFAR10,
    DataLoader,
    DataLoader,
]:
    """Load CIFAR-10 with ImageNet-style transforms.

    Returns:
        train_set, val_set, train_loader, val_loader
    """
    train_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=train_transform,
    )
    val_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=val_transform,
    )

    train_loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
    )
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, num_workers=0)

    print(
        f"CIFAR-10 (full): train={len(train_set)}, val={len(val_set)}, "
        f"classes={N_CLASSES}"
    )
    print(f"  Input size: {INPUT_SIZE}x{INPUT_SIZE} (resized for ResNet-18)")
    print(f"  Classes: {CLASS_NAMES}")

    return train_set, val_set, train_loader, val_loader


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_transfer.db"
    registry_db = "sqlite:///mlfp05_transfer_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn, artifact_store=ARTIFACT_STORE)
    return conn, tracker, "m5_transfer_learning", registry, True


def init_engines() -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# TRAINING HARNESS — shared by all technique files
# ════════════════════════════════════════════════════════════════════════


async def _train_model_async(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    tr_loader: DataLoader,
    vl_loader: DataLoader,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
) -> tuple[list[float], list[float], list[float]]:
    """Train a model and log everything to ExperimentTracker.

    Returns:
        train_losses, val_accs, train_accs (per-epoch)
    """
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in params)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"\n-- {name} --  trainable params: {n_trainable:,} / {n_total:,}")

    opt = torch.optim.Adam(params, lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    train_losses: list[float] = []
    val_accs: list[float] = []
    train_accs: list[float] = []

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "model_type": name,
                "trainable_params": str(n_trainable),
                "total_params": str(n_total),
                "epochs": str(epochs),
                "lr": str(lr),
                "batch_size": str(tr_loader.batch_size),
                "dataset_size": str(len(tr_loader.dataset)),
            }
        )

        for epoch in range(epochs):
            # -- Training --
            model.train()
            batch_losses = []
            correct = total = 0
            for xb, yb in tr_loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())
                correct += int((logits.argmax(dim=-1) == yb).sum().item())
                total += int(yb.size(0))
            train_losses.append(float(np.mean(batch_losses)))
            train_accs.append(correct / total)
            scheduler.step()

            # -- Validation --
            model.eval()
            correct = total = 0
            with torch.no_grad():
                for xb, yb in vl_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    preds = model(xb).argmax(dim=-1)
                    correct += int((preds == yb).sum().item())
                    total += int(yb.size(0))
            val_accs.append(correct / total)

            await run.log_metrics(
                {
                    "train_loss": train_losses[-1],
                    "train_acc": train_accs[-1],
                    "val_acc": val_accs[-1],
                },
                step=epoch + 1,
            )

            print(
                f"  epoch {epoch + 1}/{epochs}  "
                f"loss={train_losses[-1]:.4f}  "
                f"train_acc={train_accs[-1]:.3f}  "
                f"val_acc={val_accs[-1]:.3f}"
            )

        await run.log_metrics(
            {
                "final_val_acc": val_accs[-1],
                "best_val_acc": max(val_accs),
                "final_train_loss": train_losses[-1],
            }
        )

    return train_losses, val_accs, train_accs


def train_model(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    tr_loader: DataLoader,
    vl_loader: DataLoader,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
) -> tuple[list[float], list[float], list[float]]:
    """Sync wrapper -- one asyncio.run per training call."""
    return asyncio.run(
        _train_model_async(
            model,
            name,
            tracker,
            exp_name,
            tr_loader,
            vl_loader,
            epochs,
            lr,
        )
    )


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    val_acc: float,
    final_loss: float,
):
    """Register a trained model in the ModelRegistry."""
    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="val_acc", value=val_acc),
            MetricSpec(name="final_loss", value=final_loss),
        ],
    )
    print(f"  Registered {name}: version={version.version}, acc={val_acc:.3f}")
    return version


def register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    val_acc: float,
    final_loss: float,
):
    """Sync wrapper for model registration."""
    return asyncio.run(_register_model(registry, name, model, val_acc, final_loss))


# ════════════════════════════════════════════════════════════════════════
# SERVING HELPERS — ONNX export + InferenceServer
# ════════════════════════════════════════════════════════════════════════
IMAGE_SHAPE = (3, INPUT_SIZE, INPUT_SIZE)
N_PIXELS = 3 * INPUT_SIZE * INPUT_SIZE


class FlatImageAdapter(nn.Module):
    """Wrap an image classifier so it accepts flat pixel rows.

    InferenceServer's ONNX runtime turns each request record into ONE row of
    a 2-D float array, so the exported graph must take ``(batch, N_PIXELS)``.
    This adapter reshapes rows back to ``(batch, 3, 96, 96)`` (already
    resized and ImageNet-normalised) before calling the wrapped network.
    ``predict(X)`` is the method ``OnnxBridge.validate`` calls to get the
    native (PyTorch) outputs for comparison.
    """

    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x.reshape(-1, *IMAGE_SHAPE))

    def predict(self, X) -> np.ndarray:
        model_device = next(self.model.parameters()).device
        with torch.no_grad():
            x = torch.as_tensor(np.asarray(X), dtype=torch.float32, device=model_device)
            return self.forward(x).cpu().numpy()


def images_to_records(images: torch.Tensor) -> list[dict[str, float]]:
    """Turn a batch of preprocessed images into InferenceServer records.

    One record per image, one key per pixel. Keys are zero-padded so the
    record order is the pixel order the ONNX graph expects.
    """
    rows = images.reshape(len(images), -1).cpu().tolist()
    return [{f"px{j:05d}": float(v) for j, v in enumerate(row)} for row in rows]


def attach_onnx_artifact(name: str, version: int, onnx_path: Path) -> None:
    """Store an exported ONNX file as ``model.onnx`` for a registered version.

    ``ModelRegistry.register_model`` saves the training artifact as
    ``model.pkl``; ``InferenceServer.from_registry(..., runtime="onnx")``
    loads ``model.onnx`` for the same name + version from this ArtifactStore.

    OnnxBridge writes large weights to a sibling ``<file>.onnx.data`` file
    (ONNX external data). Storing only the ``.onnx`` bytes would leave the
    served model without its weights ("external data path does not exist"),
    so the model is loaded with its external data and stored as ONE
    self-contained protobuf.
    """
    import onnx

    model_proto = onnx.load(str(onnx_path))  # pulls in any .onnx.data weights
    asyncio.run(
        ARTIFACT_STORE.save(name, version, model_proto.SerializeToString(), "model.onnx")
    )


# ════════════════════════════════════════════════════════════════════════
# FEATURE EXTRACTION & VISUALISATION HELPERS
# ════════════════════════════════════════════════════════════════════════


def extract_features(
    model: nn.Module,
    loader: DataLoader,
    max_samples: int = 2000,
) -> tuple[np.ndarray, np.ndarray]:
    """Extract features from the penultimate layer (before fc head).

    Works for both ResNet (avgpool hook) and Sequential models.
    """
    model.eval()
    hook_features: list[torch.Tensor] = []
    labels: list[np.ndarray] = []

    def hook_fn(module, inp, out):
        del module, inp  # PyTorch hook protocol args; not used here
        hook_features.append(out.flatten(1).detach().cpu())

    # ResNet: hook into avgpool; Sequential: second-to-last layer
    if hasattr(model, "avgpool"):
        avgpool = model.avgpool
        assert isinstance(
            avgpool, nn.Module
        ), f"expected nn.Module for avgpool, got {type(avgpool).__name__}"
        handle = avgpool.register_forward_hook(hook_fn)
    else:
        assert isinstance(
            model, nn.Sequential
        ), f"hook fallback requires nn.Sequential, got {type(model).__name__}"
        handle = model[-3].register_forward_hook(hook_fn)

    with torch.no_grad():
        collected = 0
        for xb, yb in loader:
            if collected >= max_samples:
                break
            xb = xb.to(device)
            model(xb)
            labels.append(yb.numpy())
            collected += len(yb)

    handle.remove()
    features_np = torch.cat(hook_features, dim=0).numpy()[:max_samples]
    labels_np = np.concatenate(labels)[:max_samples]
    return features_np, labels_np


def compute_tsne(features: np.ndarray, perplexity: int = 30) -> np.ndarray:
    """Run t-SNE dimensionality reduction to 2D."""
    # sklearn 1.5+ renamed n_iter → max_iter in TSNE.__init__
    tsne = TSNE(n_components=2, perplexity=perplexity, max_iter=500, random_state=42)
    return tsne.fit_transform(features)


def cluster_quality(coords: np.ndarray, labels: np.ndarray) -> float:
    """Cluster quality: ratio of intra-class to inter-class distance (lower = better)."""
    intra = []
    centroids = []
    for c in range(N_CLASSES):
        mask = labels == c
        if mask.sum() < 2:
            continue
        pts = coords[mask]
        centroid = pts.mean(axis=0)
        centroids.append(centroid)
        intra.append(np.mean(np.linalg.norm(pts - centroid, axis=1)))
    centroids_arr = np.array(centroids)
    inter = np.mean(
        [
            np.linalg.norm(centroids_arr[i] - centroids_arr[j])
            for i in range(len(centroids_arr))
            for j in range(i + 1, len(centroids_arr))
        ]
    )
    avg_intra = np.mean(intra)
    return float(avg_intra / inter) if inter > 0 else float("inf")


def plot_tsne(
    coords: np.ndarray,
    labels: np.ndarray,
    title: str,
    output_path: str | Path,
) -> None:
    """Create and save a t-SNE scatter plot coloured by class."""
    fig = go.Figure()
    for c in range(N_CLASSES):
        mask = labels == c
        fig.add_trace(
            go.Scatter(
                x=coords[mask, 0],
                y=coords[mask, 1],
                mode="markers",
                name=CLASS_NAMES[c],
                marker=dict(size=4, opacity=0.6),
            )
        )
    fig.update_layout(
        title=title,
        xaxis_title="t-SNE 1",
        yaxis_title="t-SNE 2",
        template="plotly_white",
    )
    fig.write_html(str(output_path))
    print(f"  Saved: {output_path}")


def create_visualizer() -> ModelVisualizer:
    """Return a configured ModelVisualizer instance.

    ModelVisualizer carries a kailash-ml P2 experimental notice (a
    UserWarning at construction); exercises run under
    warnings-as-errors, so the notice is acknowledged narrowly at
    this construction site.
    """
    import warnings as _warnings

    from kailash_ml._decorators import ExperimentalWarning as _EW

    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore", _EW)
        return ModelVisualizer()


def save_training_plots(
    viz: ModelVisualizer,
    metrics: dict[str, list[float]],
    output_path: str | Path,
) -> None:
    """Save a training history plot to HTML."""
    fig = viz.training_history(metrics=metrics, x_label="Epoch", y_label="Value")
    fig.write_html(str(output_path))
    print(f"  Saved: {output_path}")


def classifier_diag_loss(model: nn.Module, batch) -> torch.Tensor:
    """Cross-entropy loss for ``run_diagnostic_checkpoint`` on an image classifier.

    The checkpoint passes each raw ``(images, labels)`` batch from the
    DataLoader; it does not move tensors, so we move them to the model's
    device here.
    """
    xb, yb = batch
    model_device = next(model.parameters()).device
    return F.cross_entropy(model(xb.to(model_device)), yb.to(model_device))


def count_params(model: nn.Module, trainable_only: bool = False) -> int:
    """Count parameters in a model."""
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 7, Part 4: Adapter Modules (Bridge to M6 LoRA)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this section, you will be able to:
#   - Understand why full fine-tuning is expensive and risky
#   - Build bottleneck adapter modules that inject small trainable
#     layers inside a frozen backbone
#   - Compare parameter efficiency across methods: from-scratch,
#     frozen head and adapter (LoRA follows in Module 6)
#   - Visualise the performance-vs-parameters Pareto frontier
#   - Apply adapter concepts to a multi-tenant AI platform scenario
#
# PREREQUISITES: Parts 1-3 (baseline, transfer, data efficiency).
# ESTIMATED TIME: ~25 min
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision



## THEORY — The Parameter Efficiency Spectrum


In [ ]:
# Transfer learning has a spectrum of approaches, each trading off
# between trainable parameters and model capacity:
#
# FROZEN HEAD (Part 2):
#   - Update ONLY the final classifier layer (~0.05% of params)
#   - Fastest training, zero risk of "catastrophic forgetting"
#   - Limited: the backbone features might not be perfectly suited
#
# FULL FINE-TUNING:
#   - Update ALL parameters (100% trainable)
#   - Highest potential accuracy
#   - Risk: "catastrophic forgetting" — the model unlearns ImageNet
#     features while overfitting to your small dataset
#   - Storage: need a full copy of the model per task
#
# ADAPTER MODULES (this section):
#   - Inject small trainable bottleneck layers INSIDE the frozen backbone
#   - A small fraction of params trainable (about 1% in this exercise —
#     Checkpoint 2 prints the exact share), far more than a frozen head
#   - Skip connection: adapter starts as identity, so training begins
#     from the pre-trained features
#   - Storage: only save the adapter weights (~1-5 MB) per task
#
# LoRA (Module 6):
#   - Low-rank decomposition: W + A @ B where A is d x r, B is r x d
#   - ~1-5% of params trainable — best for LLMs with billions of params
#   - Same idea as adapters but uses matrix decomposition instead of
#     bottleneck layers
#
# The key insight: you don't need to update every parameter to adapt a
# model. Most of the pre-trained knowledge is useful as-is. You only
# need to nudge the model slightly toward your specific task.



In [ ]:
print("\n" + "=" * 70)
print("  PART 4: Adapter Modules (Bridge to M6 LoRA)")
print("=" * 70)



## TASK 1 — Load data and set up engines


In [ ]:
train_set, val_set, train_loader, val_loader = load_cifar10()
conn, tracker, exp_name, registry, has_registry = init_engines()



## TASK 2 — Build the BottleneckAdapter module


The skip connection ensures the adapter starts as an identity function
    (initial up-project weights are zero), so training begins from the
    pre-trained features — not random noise.

    Architecture:
        x -> x + up_project(ReLU(down_project(x)))

    Args:
        dim: Input/output dimension (must match the layer it's inserted into)
        bottleneck: Bottleneck dimension (smaller = fewer params, less capacity)


In [ ]:
# The adapter is a small trainable module: down-project to a bottleneck
# dimension, apply a nonlinearity, then up-project back. The SKIP
# CONNECTION is critical: it means the adapter starts as an identity
# function (because up-project weights are initialised to zero), so
# training begins from the pre-trained features, not random noise.


class BottleneckAdapter(nn.Module):

    def __init__(self, dim: int, bottleneck: int = 64):
        super().__init__()
        # TODO: Create the down- and up-projection layers (self.down,
        #   self.up) and zero-initialise BOTH the weight and bias of the
        #   up-projection
        # Hint: nn.Linear for each projection; nn.init has an in-place
        #   zeros initialiser
        # Why zero-init? So the adapter starts as identity (output = input)
        ____

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Return the input plus the bottleneck path
        #   (down-project -> ReLU -> up-project)
        ____



### Checkpoint 1


In [ ]:
adapter_test = BottleneckAdapter(dim=256, bottleneck=64)
test_input = torch.randn(2, 256)
test_output = adapter_test(test_input)
# At init, the adapter is identity (zero up-projection)
assert torch.allclose(
    test_input, test_output, atol=1e-6
), "Adapter should start as identity function (zero-init up-projection)"
adapter_params = sum(p.numel() for p in adapter_test.parameters())
expected_params = 256 * 64 + 64 + 64 * 256 + 256  # down + bias + up + bias
assert (
    adapter_params == expected_params
), f"Adapter params: {adapter_params} vs expected {expected_params}"
# INTERPRETATION: The zero-init trick is crucial. Without it, the
# adapter would add random noise to the pre-trained features on the
# first forward pass, destroying the valuable ImageNet representations.
# With zero-init, training starts from "no change" and gradually learns
# task-specific adjustments.
print(f"  BottleneckAdapter(256, 64): {adapter_params:,} params, starts as identity")
print("--- Checkpoint 1 passed --- adapter module verified\n")



## TASK 3 — Build adapter-augmented ResNet-18


Wraps a ResNet block with a channel-wise adapter.


ResNet-18 with bottleneck adapters after layer3 and layer4.


In [ ]:
# We insert adapters after ResNet's layer3 and layer4 blocks. The
# adapter operates on channel-wise pooled features (spatial average
# pooling to get a vector per sample, then adapter, then broadcast
# back to spatial dimensions).


class AdaptedBlock(nn.Module):

    def __init__(self, block: nn.Module, adapter: BottleneckAdapter):
        super().__init__()
        self.block = block
        self.adapter = adapter

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Run block, then apply adapter on channel-wise pooled features
        # Steps:
        #   1. Run the wrapped block
        #   2. Average its output over the spatial dims -> pooled (B, C)
        #   3. Pass pooled through the adapter -> adapted (B, C)
        #   4. Add ONLY the adapter's change to the block output. The
        #      adapter already includes its skip connection, so adding
        #      `adapted` itself would count the pooled features twice.
        # Hint: reshape (B, C) to (B, C, 1, 1) so it broadcasts over H, W
        ____


def build_adapter_resnet(
    n_classes: int = N_CLASSES,
    bottleneck: int = 64,
) -> nn.Module:
    # No fallback to random weights: if the ImageNet download fails this
    # raises — a random frozen backbone would make the comparison meaningless.
    weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
    model = torchvision.models.resnet18(weights=weights)

    # TODO: Freeze all original parameters
    ____

    # TODO: Replace the fc head with a trainable n_classes head
    in_features = model.fc.in_features
    model.fc = ____

    # TODO: Wrap layer3 (256 channels) and layer4 (512 channels) each in an
    #   AdaptedBlock with a BottleneckAdapter of matching width
    original_layer3 = model.layer3
    original_layer4 = model.layer4
    model.layer3 = ____
    model.layer4 = ____

    return model


adapter_model = build_adapter_resnet(bottleneck=64)
adapter_model.to(device)
n_adapter_trainable = count_params(adapter_model, trainable_only=True)
n_adapter_total = count_params(adapter_model)

print(
    f"  Adapter ResNet-18: {n_adapter_trainable:,} trainable / "
    f"{n_adapter_total:,} total "
    f"({100 * n_adapter_trainable / n_adapter_total:.1f}%)"
)



### Checkpoint 2


In [ ]:
assert n_adapter_trainable > 5000, "Adapter should have more params than frozen head"
assert n_adapter_trainable < n_adapter_total, "Should have fewer trainable than total"
# Identity at init for the WHOLE wrapped block, not just the bare adapter:
# with zero-init adapters, AdaptedBlock must return exactly what the
# wrapped ResNet stage returns, so training starts from ImageNet features.
adapter_model.eval()
with torch.no_grad():
    stage_input = torch.randn(2, 128, INPUT_SIZE // 8, INPUT_SIZE // 8, device=device)
    wrapped_out = adapter_model.layer3.block(stage_input)
    adapted_out = adapter_model.layer3(stage_input)
assert torch.allclose(
    adapted_out, wrapped_out, atol=1e-6
), "AdaptedBlock should leave the wrapped stage's output unchanged at init"
# INTERPRETATION: The two adapters add 99,200 trainable parameters
# (33,088 + 66,112) on top of the 5,130-param head — about 1% of the
# ~11M total (the exact share is printed above). That is ~20x the
# frozen head's capacity to adapt, at a tiny fraction of the model.
print("--- Checkpoint 2 passed --- adapter ResNet built\n")



## TASK 4 — Train and compare all three methods


In [ ]:
print("\n" + "=" * 70)
print("  TRAINING: All three approaches on CIFAR-10")
print("=" * 70)

# Method 1: Adapter model
adapter_losses, adapter_accs, _ = train_model(
    adapter_model,
    "adapter_resnet18",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)
best_adapter = max(adapter_accs)


# Method 2: Frozen head (from Part 2)
def build_frozen_head(n_classes: int = N_CLASSES) -> nn.Module:
    # No fallback to random weights: if the ImageNet download fails this
    # raises — a random frozen backbone would make the comparison meaningless.
    # TODO: Same as Part 2 — ImageNet ResNet-18, freeze everything,
    #   fresh n_classes head; return the model
    ____


frozen_model = build_frozen_head()
frozen_losses, frozen_accs, _ = train_model(
    frozen_model,
    "frozen_head_resnet18",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)
best_frozen = max(frozen_accs)
n_frozen_trainable = count_params(frozen_model, trainable_only=True)


# Method 3: From scratch
def build_scratch_cnn(n_classes: int = N_CLASSES) -> nn.Module:
    # TODO: Return the same small CNN as Part 1
    return ____


scratch_model = build_scratch_cnn()
scratch_losses, scratch_accs, _ = train_model(
    scratch_model,
    "cnn_from_scratch",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)
best_scratch = max(scratch_accs)
n_scratch_trainable = count_params(scratch_model, trainable_only=True)



### Checkpoint 3


In [ ]:
assert (
    best_adapter > 0.20
), f"Adapter model should beat random chance (acc={best_adapter:.3f})"
assert (
    n_adapter_trainable < n_adapter_total
), "Adapter model should have fewer trainable params than total"
print(f"\n  === Parameter Efficiency Comparison ===")
print(f"  {'Method':<25} {'Trainable':>12} {'Val Acc':>10}")
print("  " + "-" * 50)
print(f"  {'From scratch':<25} {n_scratch_trainable:>12,} {best_scratch:>10.4f}")
print(f"  {'Frozen head':<25} {n_frozen_trainable:>12,} {best_frozen:>10.4f}")
print(
    f"  {'Adapter (bottleneck)':<25} {n_adapter_trainable:>12,} {best_adapter:>10.4f}"
)
print("\n--- Checkpoint 3 passed --- all three methods compared\n")



## DIAGNOSTIC CHECKPOINT — five instruments before Visualise


In [ ]:
# kailash-ml's run_diagnostic_checkpoint runs a few real forward/backward
# passes (no optimiser step) with gradient, activation and dead-neuron
# hooks attached, and replays the real per-epoch training losses. It
# RETURNS the findings; print_prescription_pad prints them. The pass
# puts the model in train mode, which updates BatchNorm running
# statistics, so we diagnose a COPY and leave the trained model intact.
import copy

from kailash_ml.diagnostics import run_diagnostic_checkpoint


print("\n── Diagnostic Report (Adapter ResNet-18 (frozen backbone + bottleneck adapters)) ──")
diag, findings = run_diagnostic_checkpoint(
    copy.deepcopy(adapter_model),
    train_loader,
    classifier_diag_loss,
    title="Adapter ResNet-18 (frozen backbone + bottleneck adapters)",
    n_batches=8,
    train_losses=adapter_losses,
    show=False,
)
print_prescription_pad(findings, "Adapter ResNet-18 (frozen backbone + bottleneck adapters)")
# HOW TO READ THE PRESCRIPTION PAD FOR THIS MODEL:
#  Gradient flow — only the adapter bottlenecks and the fc head are
#     trainable; the frozen backbone has no parameter gradients by design.
#     Read this as: do the adapter layers carry gradient? A "vanishing"
#     reading on the adapters' up-projections means they are still near
#     their zero init and barely changing the backbone's features.
#  Dead neurons — the adapters use ReLU inside the bottleneck; a high
#     inactive fraction there wastes adapter capacity (try a smaller
#     bottleneck or GELU).
#  Loss trend — read from the real per-epoch losses of the adapter run.
#  If any reading is UNKNOWN, the library could not compute it from this
#  run; the message says why.



## TASK 5 — Visualise: Parameter count vs performance Pareto chart


In [ ]:
methods = ["From Scratch", "Frozen Head", "Adapter (bottleneck=64)"]
param_counts = [n_scratch_trainable, n_frozen_trainable, n_adapter_trainable]
accuracies = [best_scratch, best_frozen, best_adapter]
colours = ["#FF5722", "#2196F3", "#4CAF50"]

fig_pareto = go.Figure()

# TODO: Scatter plot of trainable params (x) vs accuracy in % (y), one
#   labelled marker per method, coloured with `colours`
# Hint: go.Scatter with a "markers+text" mode; the layout below sets the
#   log-scale x axis
____

# (Only measured points are plotted. LoRA, the LLM-scale cousin of
# adapters, is trained and measured in Module 6.)

fig_pareto.update_layout(
    title="Parameter Efficiency: Trainable Parameters vs Validation Accuracy",
    xaxis_title="Trainable Parameters",
    yaxis_title="Validation Accuracy (%)",
    template="plotly_white",
    xaxis_type="log",
    showlegend=False,
    width=800,
    height=500,
)

pareto_path = OUTPUT_DIR / "04_parameter_pareto.html"
fig_pareto.write_html(str(pareto_path))
print(f"  Saved: {pareto_path}")

# Training curves comparison
fig_curves = go.Figure()
epochs_x = list(range(1, EPOCHS + 1))

for name, losses, accs, colour in [
    ("From Scratch", scratch_losses, scratch_accs, "#FF5722"),
    ("Frozen Head", frozen_losses, frozen_accs, "#2196F3"),
    ("Adapter", adapter_losses, adapter_accs, "#4CAF50"),
]:
    fig_curves.add_trace(
        go.Scatter(
            x=epochs_x,
            y=accs,
            mode="lines+markers",
            name=f"{name} val_acc",
            line=dict(color=colour, width=2),
        )
    )
    fig_curves.add_trace(
        go.Scatter(
            x=epochs_x,
            y=losses,
            mode="lines",
            name=f"{name} loss",
            line=dict(color=colour, width=1, dash="dot"),
        )
    )

fig_curves.update_layout(
    title="Training Curves: Three Transfer Learning Approaches",
    xaxis_title="Epoch",
    yaxis_title="Value",
    template="plotly_white",
)
curves_path = OUTPUT_DIR / "04_training_curves.html"
fig_curves.write_html(str(curves_path))
print(f"  Saved: {curves_path}")



### Checkpoint 4


In [ ]:
assert pareto_path.exists(), "Pareto chart should be saved"
assert curves_path.exists(), "Training curves should be saved"
print("--- Checkpoint 4 passed --- visualisations complete\n")



## TASK 6 — Apply: Multi-Tenant AI Platform in Singapore


In [ ]:
# SCENARIO: You're building an AI platform in Singapore that serves
# multiple clients. Each client needs a custom image classifier, but
# they share the same base architecture (ResNet-18).
#
# Full fine-tuning: store 11M params per client = ~43 MB per model
# Adapter approach: store ~100K params per client = ~0.4 MB per adapter
#
# For 50 clients, that's ~2.2 GB vs ~63 MB (one shared base + 50
# adapters). And at inference time, you can keep ONE ResNet-18 in GPU
# memory and swap adapters per request.

print("\n" + "=" * 70)
print("  APPLY: Multi-Tenant AI Platform — One Base, Many Adapters")
print("=" * 70)

N_CLIENTS = 50
FULL_MODEL_MB = n_adapter_total * 4 / (1024 * 1024)  # float32, 4 bytes each
ADAPTER_MB = n_adapter_trainable * 4 / (1024 * 1024)

print(f"\n  === Multi-Tenant Storage Analysis (50 clients) ===")
print(f"  Base model (ResNet-18): {n_adapter_total:,} params = {FULL_MODEL_MB:.1f} MB")
print(f"  Adapter weights: {n_adapter_trainable:,} params = {ADAPTER_MB:.2f} MB")
print()
print(f"  {'Approach':<30} {'Per Client':>12} {'50 Clients':>14} {'GPU Memory':>14}")
print("  " + "-" * 72)
print(
    f"  {'Full fine-tuning':<30} "
    f"{FULL_MODEL_MB:>11.1f}MB "
    f"{N_CLIENTS * FULL_MODEL_MB:>13.0f}MB "
    f"{N_CLIENTS * FULL_MODEL_MB:>13.0f}MB"
)
print(
    f"  {'Adapter (shared backbone)':<30} "
    f"{ADAPTER_MB:>11.2f}MB "
    f"{FULL_MODEL_MB + N_CLIENTS * ADAPTER_MB:>13.1f}MB "
    f"{FULL_MODEL_MB + ADAPTER_MB:>13.1f}MB"
)

# TODO: Storage saved (N full models vs one shared base + N adapters) and
#   GPU memory saved (N full models vs one base + one active adapter)
savings_storage = ____
savings_gpu = ____

print(
    f"\n  Storage savings: {savings_storage:.0f} MB ({savings_storage / (N_CLIENTS * FULL_MODEL_MB) * 100:.0f}%)"
)
print(f"  GPU memory savings: {savings_gpu:.0f} MB (load one base + swap adapters)")
print()
print(f"  INFERENCE COST COMPARISON:")
print(f"  Full fine-tuning: 50 separate models, each needing GPU memory")
print(f"    -> Need multiple GPUs or sequential loading (slow)")
print(f"  Adapter approach: 1 base model in GPU + swap {ADAPTER_MB:.2f} MB adapters")
print(
    f"    -> Single GPU serves all 50 clients with ~{ADAPTER_MB * 1000:.0f}KB swap per request"
)
print()
print(f"  BRIDGE TO M6:")
print(f"  In M6, you will learn LoRA (Low-Rank Adaptation) — the adapter")
print(f"  technique designed for LLMs with BILLIONS of parameters.")
print(f"  Same concept: inject small trainable matrices into frozen weights.")
print(f"  LoRA uses low-rank decomposition: W + A @ B where rank r << d.")
print(f"  A GPT-scale model with 7B params might need only ~4M LoRA params (0.06%).")



### Checkpoint 5


In [ ]:
assert savings_storage > 0, "Adapter approach should save storage"
assert ADAPTER_MB < FULL_MODEL_MB, "Adapter should be smaller than full model"
# INTERPRETATION: Adapters make multi-tenant ML serving economically
# viable. Instead of N copies of the full model (one per client), you
# store one shared backbone plus N tiny adapter files. This is the
# same principle that makes LoRA practical for LLM fine-tuning in M6.
print("\n--- Checkpoint 5 passed --- multi-tenant analysis complete\n")



## CLEANUP


In [ ]:
await conn.close()



## REFLECTION


[x] Built a BottleneckAdapter module with zero-init skip connection
  [x] Inserted adapters into frozen ResNet-18 (layer3 + layer4)
  [x] Compared three approaches:
      From scratch: {n_scratch_trainable:>8,} params -> {best_scratch:.1%} accuracy
      Frozen head:  {n_frozen_trainable:>8,} params -> {best_frozen:.1%} accuracy
      Adapter:      {n_adapter_trainable:>8,} params -> {best_adapter:.1%} accuracy
  [x] Visualised the parameter-efficiency Pareto frontier
  [x] Analysed multi-tenant serving: {savings_storage:.0f} MB saved across 50 clients

  THE TRANSFER LEARNING SPECTRUM:
    Frozen head  ->  Adapter/LoRA  ->  Partial fine-tune  ->  Full fine-tune
    (fewest params)                                         (most params)
    (fastest training)                                      (highest capacity)
    (safest from forgetting)                                (risk of forgetting)

  BRIDGE TO M6 — LoRA:
    Adapters: bottleneck layers (down -> ReLU -> up)
    LoRA:     low-rank matrices (W + A @ B, where r << d)
    Same principle, different math. LoRA is preferred for LLMs because
    the low-rank decomposition is more parameter-efficient at scale.

  NEXT: Part 5 deploys the best model to production with ONNX export
  and InferenceServer — the full pipeline from experiment to serving.


In [ ]:
print("\n" + "=" * 70)
print("  PART 4 COMPLETE — What You've Learned")
print("=" * 70)
print(
)

